# Builder Design Pattern

implemented in a modern, Pythonic way (using Method Chaining and Dataclasses).

#### The Concept

The Builder Pattern is used to construct complex objects step-by-step. Unlike the Factory (which creates an object in one shot), the Builder allows you to configure an object with various options before finally "building" it.

Think of it like ordering a custom sandwich (sub):
- Choose Bread
- Choose Cheese
- Add Veggies
- Add Sauce
- Wrap it up (Build)

## The Classic OOP Way (Java-Style)

In the strict OOP approach, the `Computer` object is immutable (once built, it doesn't change). We create a separate helper class `ComputerBuilder` to hold the temporary state while we configure the machine.

We can achieve immutibility with `@dataclass(frozen=True)`

#### THE PRODUCT (Complex Object)

In [29]:
class Computer:
    def __init__(self, cpu, ram, storage, gpu, bluetooth, cooling):
        # In Java, these would often be private with getters
        self.cpu = cpu
        self.ram = ram
        self.storage = storage
        self.gpu = gpu
        self.bluetooth = bluetooth
        self.cooling = cooling

    def __str__(self) -> str:
        return (f"🖥️  Computer Specs:\n"
                f"   ├── CPU:           {self.cpu}\n"
                f"   ├── RAM:           {self.ram}\n"
                f"   ├── Storage:       {self.storage}\n"
                f"   ├── GPU:           {self.gpu or 'Integrated Graphics'}\n"
                f"   ├── bluetooth:     {self.bluetooth}\n"
                f"   └── Cooling:       {self.cooling}")

## THE BUILDER

In [30]:
from typing import Self

class ComputerBuilder:
    """
    The Builder allows step-by-step construction.
    Methods return 'Self' to allow Method Chaining (fluent interface).
    """
    def __init__(self):
        # Set defaults for optional parameters
        self._cpu = "Default CPU"
        self._ram = "Default RAM"
        self._storage = "Default HDD"
        self._gpu = None
        self._bluetooth = False
        self._cooling = "Air"

    # --- Chainable Methods (The "Fluent" Interface) ---
    def set_cpu(self, cpu):
        self._cpu = cpu
        return self # Return self to allow chaining

    def set_ram(self, ram):
        self._ram = ram
        return self

    def set_storage(self, storage):
        self._storage = storage
        return self

    def set_gpu(self, gpu):
        self._gpu = gpu
        return self

    def enable_bluetooth(self):
        self._bluetooth = True
        return self

    def set_liquid_cooling(self):
        self._cooling = "Liquid"
        return self

    def build(self) -> Computer:
        """Finalizes construction and returns the object."""
        # You could add validation logic here (e.g., check compatibility)
        return Computer(
            self._cpu, 
            self._ram, 
            self._storage, 
            self._gpu, 
            self._bluetooth, 
            self._cooling
        )

## CLIENT CODE

In [31]:
def main():
    print("--- Java-Style Builder ---")
    
    # We construct it step-by-step. 
    # Notice how readable this is compared to a massive constructor.
    my_gaming_pc = (ComputerBuilder()
                    .set_cpu("Intel i9")
                    .set_ram("32GB")
                    .set_storage("1TB NVMe")
                    .set_gpu("RTX 4090")
                    .set_liquid_cooling()
                    .build())

    print(my_gaming_pc)

    # Building a simple office PC
    office_pc = (ComputerBuilder()
                 .set_cpu("Intel i5")
                 .set_ram("16GB")
                 .build()) # Uses defaults for the rest
                 
    print(office_pc)

if __name__ == "__main__":
    main()

--- Java-Style Builder ---
🖥️  Computer Specs:
   ├── CPU:           Intel i9
   ├── RAM:           32GB
   ├── Storage:       1TB NVMe
   ├── GPU:           RTX 4090
   ├── bluetooth:     False
   └── Cooling:       Liquid
🖥️  Computer Specs:
   ├── CPU:           Intel i5
   ├── RAM:           16GB
   ├── Storage:       Default HDD
   ├── GPU:           Integrated Graphics
   ├── bluetooth:     False
   └── Cooling:       Air


#### Why is this Pythonic?

- `typing.Self`: Introduced in Python 3.11, this allows the type checker to know that `set_cpu()` returns the Builder instance itself, enabling the nice dot-notation chain `(.set_cpu().set_ram())`.
- Dataclasses: The `Product` is a simple dataclass, removing the need for a huge `__init__` method full of boilerplate assignments.
- Method Chaining: This mimics popular Python libraries (like Pandas or SQLAlchemy) where you chain operations together.

#### When to use this?

- Use it when: Your object has 5+ parameters, many of which are optional, or if the construction process involves complex logic (like "if you add a GPU, you must upgrade the Power Supply").
- Don't use it when: Your object only needs 2-3 arguments. Just use `Computer(cpu="i7", ram="16GB") `(standard constructor).
- Would you like to see the Singleton Pattern next (ensuring only one instance of a class exists)?

## The Pythonic Way

In Python, we usually **do not implement the Builder Pattern** explicitly because Python supports **Named Arguments** and **Default Parameter Values** natively. The "Telescoping Constructor" problem simply doesn't exist in Python. We can get the same readability and optional parameters using a simple class constructor or a `@dataclass`.

#### The Truth About Python Builders
In languages like Java or C++, the Builder pattern is necessary because constructors cannot easily handle optional parameters (you end up with "Telescoping Constructors").

In Python, we have Named Arguments and Default Values. </br>
Therefore, the most Pythonic Builder is usually just the Class itself (specifically a dataclass). It is cleaner, faster, and 100% built-in.

#### THE PYTHONIC "BUILDER" (Just the Class!)

In [35]:
from dataclasses import dataclass, field
from typing import Optional

@dataclass(frozen=True)
class Computer:
    """
    In Python, the 'Builder' pattern is built into the language features:
    1. Keyword Arguments (naming parameters)
    2. Default Values (optional parameters)
    """
    # Required parts (Must be provided)
    cpu: str
    
    # Optional parts (Have defaults)
    ram: str = "8GB"
    storage: str = "256GB SSD"
    gpu: Optional[str] = None
    bluetooth: bool = False
    cooling: str = "Air Cooling"

    # Derived/Computed fields (Logic that usually lives in a Builder 'build()' method)
    # can live in __post_init__. Since we used frozen=True, we have a workaround,
    # or simpler: just use a property.
    @property
    def specs(self) -> str:
        return (f"🖥️  Computer Specs:\n"
                f"   ├── CPU:           {self.cpu}\n"
                f"   ├── RAM:           {self.ram}\n"
                f"   ├── Storage:       {self.storage}\n"
                f"   ├── GPU:           {self.gpu or 'Integrated Graphics'}\n"
                f"   ├── bluetooth:     {self.bluetooth}\n"
                f"   └── Cooling:       {self.cooling}")

    def __str__(self):
        return self.specs

    # ==========================================
    # 2. "DIRECTOR" LOGIC (Factory Methods)
    # ==========================================
    # Instead of a separate 'Director' class, use Class Methods.
    
    @classmethod
    def create_gaming_rig(cls) -> "Computer":
        """Pre-configured Gaming Setup"""
        return cls(
            cpu="Intel Core i9",
            ram="32GB DDR5",
            gpu="NVIDIA RTX 4090",
            cooling="Liquid Loop",
            bluetooth=True,
            storage="2TB NVMe"
        )

    @classmethod
    def create_office_pc(cls) -> "Computer":
        """Pre-configured Office Setup"""
        return cls(
            cpu="Intel Core i5",
            ram="16GB"
            # Uses default storage, gpu, cooling
        )

### CLIENT CODE

In [36]:
def main():
    print("--- Pythonic Way ---")

    # 1. Complex Build
    # Look how similar this is to the Builder pattern, but without the extra class.
    # We explicitly name the arguments we care about.
    gaming_pc = Computer(
        cpu="AMD Ryzen 9",
        ram="64GB",
        gpu="RTX 4080",
        cooling="Liquid",
        bluetooth=True,
        storage="2TB SSD"
    )

    # 2. Simple Build
    # We skip the optional ones, and Python handles the defaults.
    office_pc = Computer(
        cpu="Intel i3",
        ram="8GB"
    )
    
    # 3. Out of order? No problem.
    server = Computer(
        ram="128GB",
        cpu="Threadripper",
        cooling="Fan Wall"
    )

    print(gaming_pc)
    print(office_pc)
    print(server)

if __name__ == "__main__":
    main()

--- Pythonic Way ---
🖥️  Computer Specs:
   ├── CPU:           AMD Ryzen 9
   ├── RAM:           64GB
   ├── Storage:       2TB SSD
   ├── GPU:           RTX 4080
   ├── bluetooth:     True
   └── Cooling:       Liquid
🖥️  Computer Specs:
   ├── CPU:           Intel i3
   ├── RAM:           8GB
   ├── Storage:       256GB SSD
   ├── GPU:           Integrated Graphics
   ├── bluetooth:     False
   └── Cooling:       Air Cooling
🖥️  Computer Specs:
   ├── CPU:           Threadripper
   ├── RAM:           128GB
   ├── Storage:       256GB SSD
   ├── GPU:           Integrated Graphics
   ├── bluetooth:     False
   └── Cooling:       Fan Wall


#### Key Differences

| Feature         | Classic OOP                                                          | Pythonic                                                     |
|-----------------|----------------------------------------------------------------------|----------------------------------------------------------------|
| **Complexity**  | High — requires two classes (`Computer` and `ComputerBuilder`).      | Low — single class (`Computer`).                               |
| **Mechanism**   | Setter methods (`set_cpu`, `set_ram`) store temporary state.          | `__init__` arguments (`cpu=...`) handle state directly.       |
| **Readability** | `builder.set_cpu("i9").build()`                                      | `Computer(cpu="i9")`                                          |
| **Validation**  | Validation logic placed in `build()` (e.g., compatibility checks).   | Validation logic in `__init__` or `__post_init__` (dataclasses). |


#### When to actually use a Builder Class in Python?

Use the Java-style Builder in Python ONLY if:
- **Construction is extremely complex**: e.g., Parsing a SQL query where you need to validate that `GROUP BY` columns exist in the `SELECT` clause before building the query object.
- **Chained steps are required**: e.g., A query builder like `.select().where().limit()`.
- **Immutability is strict**: You want the final object to be frozen, but you need a mutable object to set it up.

# Builder Design Pattern 

explained using a complex, real-world software engineering scenario: A SQL Query Builder.

#### The Scenario: Dynamic SQL Construction

In enterprise software (`like ORMs - Hibernate, SQLAlchemy`), you rarely write raw SQL strings like `"SELECT * FROM users WHERE..."`. Instead, the code constructs queries dynamically based on user input.

### Complexity:
- You might add multiple `WHERE` clauses based on different `if` conditions (e.g., if user has a filter, add it).
- You might have optional `JOIN`s.
- Structure matters: You cannot have `HAVING` without `GROUP BY`.
- State is **Cumulative**: You don't set the "Where" clause once; you might append 5 different constraints to it over the lifecycle of the request.

This makes the "Named Arguments" approach (used in the simple Computer example) insufficient, because we need to accumulate state step-by-step.

## The Classic OOP Way (Java-Style)

In the strict OOP approach, we separate the Product (The immutable SQL string/object) from the **Builder** (The logic that assembles it). The Product is often "dumb" (just holds data), while the Builder handles the list management and string concatenation logic.

#### THE PRODUCT (Immutable Result)

In [37]:
class SqlQuery:
    """
    The final immutable object that represents the query.
    """
    def __init__(self, content: str):
        self._content = content

    def execute(self):
        # Simulate sending to DB
        print(f"⚡ Executing SQL: {self._content}")

#### THE BUILDER

from typing import List

class SqlQueryBuilder:
    def __init__(self):
        # We need Lists because these are cumulative
        self._table = ""
        self._columns: List[str] = []
        self._wheres: List[str] = []
        self._joins: List[str] = []
        self._limit = None

    # --- Step 1: Mandatory Step ---
    def from_table(self, table: str):
        self._table = table
        return self

    # --- Step 2: Cumulative Steps ---
    def select(self, column: str):
        self._columns.append(column)
        return self

    def where(self, condition: str):
        self._wheres.append(condition)
        return self

    def join(self, table: str, on_condition: str):
        self._joins.append(f"JOIN {table} ON {on_condition}")
        return self

    def limit(self, count: int):
        self._limit = count
        return self

    # --- Step 3: The Build Method ---
    def build(self) -> SqlQuery:
        # 1. Validation Logic
        if not self._table:
            raise ValueError("SQL Error: Table name is missing!")
        
        if not self._columns:
            self._columns.append("*") # Default to Select All

        # 2. String Assembly
        query_str = f"SELECT {', '.join(self._columns)} FROM {self._table}"

        if self._joins:
            query_str += " " + " ".join(self._joins)

        if self._wheres:
            query_str += " WHERE " + " AND ".join(self._wheres)

        if self._limit:
            query_str += f" LIMIT {self._limit}"

        query_str += ";"
        
        # 3. Return Product
        return SqlQuery(query_str)

#### CLIENT CODE

In [39]:
def main():
    print("--- Java-Style SQL Builder ---")
    
    # Imagine these come from UI filters
    user_filters = {"age": 18, "status": "active"}

    builder = SqlQueryBuilder()
    builder.from_table("users")
    builder.select("id")
    builder.select("username")
    
    # Dynamic Logic Application
    if "age" in user_filters:
        builder.where(f"age > {user_filters['age']}")
    
    if "status" in user_filters:
        builder.where(f"status = '{user_filters['status']}'")

    # Finalize
    query = builder.build()
    query.execute()

if __name__ == "__main__":
    main()

--- Java-Style SQL Builder ---
⚡ Executing SQL: SELECT id, username FROM users WHERE age > 18 AND status = 'active';


## The Pythonic Way (Fluent Interface & Magic Methods)

In Python, we often merge the **Product** and the **Builder** into a single class if the construction logic is the main purpose of the object (like in ORMs).
- `__str__`: Instead of a `.build()` method, we use the magic string representation method. The object builds itself when you try to print it.
- `*args`: We use variable arguments to make `select("a", "b", "c")` cleaner than `select(["a", "b", "c"])`.
- **Chaining**: We return self to allow the fluent syntax common in Python libraries (like Pandas or SQLAlchemy).

#### THE PYTHONIC BUILDER (Merged Product)

In [41]:
from typing import List, Optional

class Query:
    def __init__(self, table: str):
        # We start with the table, creating a cleaner init
        self.table = table
        self.columns: List[str] = []
        self.conditions: List[str] = []
        self.limit_val: Optional[int] = None

    # Use *args for cleaner syntax: .select("id", "name")
    def select(self, *args):
        self.columns.extend(args)
        return self # Enable chaining

    def where(self, condition: str):
        self.conditions.append(condition)
        return self

    def limit(self, val: int):
        self.limit_val = val
        return self

    # MAGIC METHOD: Acts as the 'build()' method automatically
    def __str__(self):
        # Logic to compile the query
        cols = ", ".join(self.columns) if self.columns else "*"
        
        sql = f"SELECT {cols} FROM {self.table}"
        
        if self.conditions:
            sql += " WHERE " + " AND ".join(self.conditions)
            
        if self.limit_val:
            sql += f" LIMIT {self.limit_val}"
            
        return sql + ";"

    def execute(self):
        # In a real app, this would connect to the DB driver
        print(f"🚀 Running: {self}")

#### CLIENT CODE

In [42]:
def main():
    print("--- Pythonic SQL Builder ---")

    # 1. Fluent Chaining
    # Notice how we don't need a separate "Builder" class instance.
    # The Query object IS the builder.
    q = (Query("orders")
         .select("order_id", "date", "customer_id")
         .where("amount > 100")
         .where("status = 'PAID'")
         .limit(50))

    # 2. The Build happens automatically when we treat it as a string
    print(f"Generated SQL: {q}")
    
    # 3. Dynamic Injection (Common in API endpoints)
    request_params = {"region": "US", "vendor": "Acme"}
    
    q2 = Query("sales").select("total")
    
    for key, val in request_params.items():
        q2.where(f"{key} = '{val}'")

    q2.execute()

if __name__ == "__main__":
    main()

--- Pythonic SQL Builder ---
Generated SQL: SELECT order_id, date, customer_id FROM orders WHERE amount > 100 AND status = 'PAID' LIMIT 50;
🚀 Running: SELECT total FROM sales WHERE region = 'US' AND vendor = 'Acme';


#### Why "Named Arguments" (Previous Example) failed here

In the previous Computer example, we used `Computer(cpu="Intel", ram="16GB")`. That worked because `cpu` is set once. In this SQL example, we cannot use `Query(where="x", where="y")` because keyword arguments must be unique. When you need cumulative state (adding multiple items to a list over time), you must use methods (`.where()`) or a Class-based Builder pattern.

#### Real-World Python Examples

SQLAlchemy / Django ORM:
```py
# This is the Builder Pattern in action!
User.objects.filter(name='Alice').exclude(age__lt=18).order_by('-created_at')
```

Matplotlib
```py
# Building a complex plot step-by-step
plt.figure().add_subplot().plot(data).set_title("Builder Pattern")
```